# Set up Google Earth Engine for this course

<a target="_blank" href="https://colab.research.google.com/github/khouakhi/UMP_EO_training/blob/main/notebooks/00_setup_google_earth_engine.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>


## Why this notebook?

Earth Observation (EO) data for this module live in **Google Earth Engine (GEE)**. Before you can load rainfall, satellite images, or maps, you must:

1. Have a **Google account** accepted on the Earth Engine platform.
2. Tell the Python library **which GEE cloud project** to bill and store assets under.

Use your own cloud project ID.

## What you will do

- Install the Python libraries used in all other notebooks.
- Run a one-line **authentication** action (browser sign-in).
- **Initialise** Earth Engine with your project.
- Run a **tiny test**: load one public dataset and print a statistic for Morocco.

---

### Before you start (first time only)

1. Go to [Earth Engine signup](https://earthengine.google.com/signup/) if you have never used GEE before.
2. Wait until you can open the [Earth Engine Code Editor](https://code.earthengine.google.com/) without errors.

When those work, continue below.


## 1. Install libraries

Run the cell below. On Google Colab this takes about a minute the first time.


In [ ]:
# Install packages (Colab often needs a fresh install each session)
!pip install -q earthengine-api geemap

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

# If you run this notebook locally, run `ee.Authenticate()` once before `ee.Initialize`.
# If the Colab pop-up fails, try: ee.Authenticate(auth_mode="colab")
# Mapping notebooks use `Map.add_basemap("SATELLITE")` so you always have photo context under EE layers.


## 2. Authenticate and initialise

**Authenticate:** the first time in a new environment, Earth Engine opens a browser window. Approve access for the account that has been added to your project.

**Initialise:** set `EE_PROJECT` to your own cloud project ID, then run `ee.Initialize(project=...)` so all requests use that project.




In [ ]:
# Connect to Google Earth Engine using your cloud project ID.
# Set this to your own Google Earth Engine cloud project ID before running.
EE_PROJECT = "YOUR_GEE_PROJECT_ID"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine initialised with project:", EE_PROJECT)


## 3. Small test (CHIRPS rainfall)

We load **December 2025** daily [**CHIRPS Daily**](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY) (the same product used later) and compute the **mean grid-cell total rainfall** over Morocco for that month (millimetres summed over the month, then averaged spatially). This matches the course focus on the **extended winter-spring wet season** (December-April).

If you see a number printed (not an error), your setup is working.


In [ ]:
# Quick test: December 2025 total CHIRPS rainfall (mm), spatial mean over Morocco
morocco = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Morocco"))

chirps = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterDate("2025-12-01", "2026-01-01")
    .sum()
)

mean_mm = chirps.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=morocco.geometry(),
    scale=5500,
    maxPixels=1e13,
    tileScale=4,
).get("precipitation")

print("Mean December 2025 total rainfall over Morocco (mm, approximate):", mean_mm.getInfo())


## Suggested timing for the whole 3-hour practical

| Block (minutes) | Notebook | Focus |
|---:|---|---|
| 0-15 | `00` | Setup and authentication |
| 15-45 | `01` | AOI and landscape context |
| 45-80 | `02` | CHIRPS rainfall totals and anomaly |
| 80-120 | `03` | Surface water (Sentinel-2 / optional Sentinel-1) |
| 120-155 | `04` | NDVI and green-up |
| 155-200 | `05` | RF ponds from markers (binary) |
| 200-210 | `-` | Group conclusion paragraph |

Notebook `05` now contains the full ponds workflow.

---

## Checklist before the next notebook

- [ ] `ee.Initialize` ran without errors.
- [ ] The [**CHIRPS**](https://developers.google.com/earth-engine/datasets/catalog/UCSB-CHG_CHIRPS_DAILY) test printed a rainfall value.
- [ ] You know which Google account you used.

**Next:** open `01_aoi_explore_lower_moulouya.ipynb` to map the study area.
